In [1]:
import pandas as pd

file_path = "inputs.xlsx"
df = pd.read_excel(file_path)
df

,link,id,fundingSource,institute,isHumanStudy,consentDescription,element_1A
0,https://grants.nih.gov/sites/default/files/flm...,1,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,"Demographic, clinical, and MRI, 1 H fMRS and f..."
1,https://grants.nih.gov/sites/default/files/flm...,2,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,Our genomic study will be registered with dbGa...
2,https://grants.nih.gov/sites/default/files/flm...,3,NIH,National Institute of Mental Health (NIMH),no,no,"As detailed in the Research Strategy Section, ..."
3,https://grants.nih.gov/sites/default/files/flm...,4,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,The data to be shared will include MRI images ...


In [2]:
def build_prompt(row):
    funding_source = str(row.get("fundingSource", "Not provided")).strip()
    institute = str(row.get("institute", "Not provided")).strip()
    is_human = str(row.get("isHumanStudy", "No")).strip()
    consent = str(row.get("consentDescription", "no")).strip()
    data_collected = str(row.get("element_1A", "Not provided")).strip()

    consent_block = ""
    if is_human.lower() in ["yes", "true", "1"]:
        consent_block = f"\n- If yes, data sharing consent: {consent if consent else 'Not provided'}"

    prompt = f"""
You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to {funding_source}, specifically targeting {institute}.

Proposal context:
- Funding source: {funding_source}
- Human subjects study: {is_human}{consent_block}
- Data to be collected: {data_collected}

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the following sections:

1- Suggested file formats for each data type
Identify the recommended file format or formats for each data type in the dataset. Briefly explain why these formats are appropriate for FAIR sharing, reuse, interoperability, and long-term preservation.
2- Tools for converting data to the suggested file formats
List tools, software, or workflows that can be used to convert raw or source data into the recommended file formats.
3- Suggested standard structure to organize the dataset
State the recommended dataset structure, schema, specification, or community standard that should be used to organize the dataset, if one exists.
4- Tools for organizing the dataset in the suggested structure
List tools, software, templates, or platforms that can help organize the dataset according to the recommended structure or standard.
5- Metadata and information that should be provided with the data
Describe the metadata, documentation, contextual information, data dictionaries, codebooks, README files, protocols, and any other supporting information that should accompany the dataset.
6- Suggested standard structure and file format for metadata
State the recommended structure, schema, standard, and file format that should be used for metadata and documentation files included with the dataset.
7-Tools for preparing metadata in the suggested structure and file format
List tools, software, validators, templates, or platforms that can help create, manage, standardize, or validate metadata in the recommended structure and format.
8- Suggested de-identification approach
State whether de-identification is required. Explain the recommended de-identification or privacy-preserving approach based on the subject type, data modality, sensitivity of the data, and whether human participants are involved. If consent language affects sharing, explain that clearly.
9- Tools to implement the suggested de-identification approach
List tools, software, workflows, or methods that can help implement the recommended de-identification or privacy protection approach.
10-Suggested repositories for sharing the dataset
Recommend appropriate repositories for sharing the dataset, prioritizing repositories required, recommended, or commonly accepted by the funding source, subject type, and data modality.
11- Suggested license for sharing the dataset
Recommend an appropriate data license or licensing approach, prioritizing what is expected, suggested, or required by the funding source. If restrictions apply because of human subjects, controlled access, or consent limitations, explain that clearly.

Instructions for your response:
- Make the guidance practical, concise, and easy for researchers to follow.
- Tailor all recommendations to the combination of funding source, human-subject status, consent language, and data modality.
- When funder policies, privacy, or consent restrictions affect the answer, explain that explicitly.
- When no single standard or repository is certain, label the answer as “Recommended approach” and provide the best evidence-based guidance.
- Prefer community-accepted standards, open formats, and reusable metadata practices whenever possible.
- Keep the tone professional, helpful, and grant-writing appropriate.
"""
    return prompt.strip()


# Create prompt column
df["prompt"] = df.apply(build_prompt, axis=1)

# Show first prompt
print(df["prompt"].iloc[0])

You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the

In [3]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask_gpt(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a FAIR data expert helping generate practical FAIR instructions."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content

# Apply to each row
df["gpt_answer"] = df["prompt"].apply(ask_gpt)

# Save output
df.to_excel("FAIR_instructions_with_prompts_and_answers1.xlsx", index=False)

In [4]:
for i, row in df.iterrows():
    print(f"\n--- Row {i} ---")
    print("PROMPT:\n", row["prompt"])
    print("\nANSWER:\n", row["gpt_answer"])
    print("\n" + "="*60)


--- Row 0 ---
PROMPT:
 You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guid

# Another prompt version

In [6]:
import json
import pandas as pd

def build_prompt(row):
    funding_source = str(row.get("fundingSource", "Not provided")).strip()
    institute = str(row.get("institute", "Not provided")).strip()
    is_human = str(row.get("isHumanStudy", "No")).strip()
    consent = str(row.get("consentDescription", "no")).strip()
    data_collected = str(row.get("element_1A", "Not provided")).strip()

    consent_block = ""
    if is_human.lower() in ["yes", "true", "1"]:
        consent_block = f"\n- If yes, data sharing consent: {consent if consent else 'Not provided'}"

    prompt = f"""
You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to {funding_source}, specifically targeting {institute}.

Proposal context:
- Funding source: {funding_source}
- Human subjects study: {is_human}{consent_block}
- Data to be collected: {data_collected}

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the following sections:

1- Suggested file formats for each data type
Identify the recommended file format or formats for each data type in the dataset. Briefly explain why these formats are appropriate for FAIR sharing, reuse, interoperability, and long-term preservation.
2- Tools for converting data to the suggested file formats
List tools, software, or workflows that can be used to convert raw or source data into the recommended file formats.
3- Suggested standard structure to organize the dataset
State the recommended dataset structure, schema, specification, or community standard that should be used to organize the dataset, if one exists.
4- Tools for organizing the dataset in the suggested structure
List tools, software, templates, or platforms that can help organize the dataset according to the recommended structure or standard.
5- Metadata and information that should be provided with the data
Describe the metadata, documentation, contextual information, data dictionaries, codebooks, README files, protocols, and any other supporting information that should accompany the dataset.
6- Suggested standard structure and file format for metadata
State the recommended structure, schema, standard, and file format that should be used for metadata and documentation files included with the dataset.
7-Tools for preparing metadata in the suggested structure and file format
List tools, software, validators, templates, or platforms that can help create, manage, standardize, or validate metadata in the recommended structure and format.
8- Suggested de-identification approach
State whether de-identification is required. Explain the recommended de-identification or privacy-preserving approach based on the subject type, data modality, sensitivity of the data, and whether human participants are involved. If consent language affects sharing, explain that clearly.
9- Tools to implement the suggested de-identification approach
List tools, software, workflows, or methods that can help implement the recommended de-identification or privacy protection approach.
10-Suggested repositories for sharing the dataset
Recommend appropriate repositories for sharing the dataset, prioritizing repositories required, recommended, or commonly accepted by the funding source, subject type, and data modality.
11- Suggested license for sharing the dataset
Recommend an appropriate data license or licensing approach, prioritizing what is expected, suggested, or required by the funding source. If restrictions apply because of human subjects, controlled access, or consent limitations, explain that clearly.

Instructions for your response:
- Make the guidance practical, concise, and easy for researchers to follow.
- Tailor all recommendations to the combination of funding source, human-subject status, consent language, and data modality.
- When funder policies, privacy, or consent restrictions affect the answer, explain that explicitly.
- When no single standard or repository is certain, label the answer as "Recommended approach" and provide the best evidence-based guidance.
- Prefer community-accepted standards, open formats, and reusable metadata practices whenever possible.
- Keep the tone professional, helpful, and grant-writing appropriate.

Return your answer in valid JSON format only.


Use exactly this JSON structure:
{{
  "1-Suggested file formats for each data type": "",
  "2-Tools for converting data to the suggested file formats": "",
  "3-Suggested standard structure to organize the dataset": "",
  "4-Tools for organizing the dataset in the suggested structure": "",
  "5-Metadata and information that should be provided with the data": "",
  "6-Suggested standard structure and file format for metadata": "",
  "7-Tools for preparing metadata in the suggested structure and file format": "",
  "8-Suggested de-identification approach": "",
  "9-Tools to implement the suggested de-identification approach": "",
  "10-Suggested repositories for sharing the dataset": "",
  "11-Suggested license for sharing the dataset": ""
}}
"""
    return prompt.strip()


# Create prompt column
df["prompt"] = df.apply(build_prompt, axis=1)

# Show first prompt
print(df["prompt"].iloc[0])

You are a FAIR data expert helping researchers make their data more Findable, Accessible, Interoperable, and Reusable (FAIR).

Your task is to generate practical FAIR data instructions for a grant proposal being submitted to NIH, specifically targeting National Institute of Mental Health (NIMH).

Proposal context:
- Funding source: NIH
- Human subjects study: yes
- If yes, data sharing consent: All research participants will be consented for broad data sharing.
- Data to be collected: Demographic, clinical, and MRI, 1 H fMRS and fMRI imaging data will be acquired from 110 affected youth and 110 matched healthy controls (described in detail in sections C.3 and C.4 of this application). All data will be deidentified prior to receipt by the repository, but the information needed to generate a global unique identifier for the NIMH Data Archive (NDA) will be collected for each subject.

Based on these inputs, provide clear, practical, and user-friendly FAIR data guidance organized under the

In [7]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask_gpt(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a FAIR data expert helping generate practical FAIR instructions."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content

# Apply to each row
df["gpt_answer"] = df["prompt"].apply(ask_gpt)

# Save output
df.to_excel("FAIR_instructions_with_prompts_and_answers2.xlsx", index=False)

In [8]:
import json
import re
import pandas as pd

def parse_json_result(text):
    empty_result = {
  "1-Suggested file formats for each data type": "",
  "2-Tools for converting data to the suggested file formats": "",
  "3-Suggested standard structure to organize the dataset": "",
  "4-Tools for organizing the dataset in the suggested structure": "",
  "5-Metadata and information that should be provided with the data": "",
  "6-Suggested standard structure and file format for metadata": "",
  "7-Tools for preparing metadata in the suggested structure and file format": "",
  "8-Suggested de-identification approach": "",
  "9-Tools to implement the suggested de-identification approach": "",
  "10-Suggested repositories for sharing the dataset": "",
  "11-Suggested license for sharing the dataset": ""
}

    try:
        if pd.isna(text):
            return pd.Series(empty_result)

        text = str(text).strip()

        # If GPT added extra text before/after JSON, extract only the JSON part
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            text = match.group(0)

        data = json.loads(text)

        return pd.Series({
            "1-Suggested file formats for each data type": data.get("1-Suggested file formats for each data type", ""),
            "2-Tools for converting data to the suggested file formats": data.get("2-Tools for converting data to the suggested file formats", ""),
            "3-Suggested standard structure to organize the dataset": data.get("3-Suggested standard structure to organize the dataset", ""),
            "4-Tools for organizing the dataset in the suggested structure": data.get("4-Tools for organizing the dataset in the suggested structure", ""),
            "5-Metadata and information that should be provided with the data": data.get("5-Metadata and information that should be provided with the data", ""),
            "6-Suggested standard structure and file format for metadata": data.get("6-Suggested standard structure and file format for metadata", ""),
            "7-Tools for preparing metadata in the suggested structure and file format": data.get("7-Tools for preparing metadata in the suggested structure and file format", ""),
            "8-Suggested de-identification approach": data.get("8-Suggested de-identification approach", ""),
            "9-Tools to implement the suggested de-identification approach": data.get("9-Tools to implement the suggested de-identification approach", ""),
            "10-Suggested repositories for sharing the dataset": data.get("10-Suggested repositories for sharing the dataset", ""),
            "11-Suggested license for sharing the dataset": data.get("11-Suggested license for sharing the dataset", "")
        })

    except Exception:
        return pd.Series(empty_result)


# Keep only your original columns first
base_cols = [
    "link",
    "id",
    "fundingSource",
    "institute",
    "isHumanStudy",
    "consentDescription",
    "element_1A",
    "prompt",
    "gpt_answer"
]

df_base = df[base_cols].copy()

# Parse JSON from gpt_answer
parsed_cols = df_base["gpt_answer"].apply(parse_json_result)

# Combine original columns + parsed columns only once
df_clean = pd.concat([df_base, parsed_cols], axis=1)

# Preview selected columns
print(df_clean[[
    "1-Suggested file formats for each data type",
    "2-Tools for converting data to the suggested file formats",
    "3-Suggested standard structure to organize the dataset"
]].head())

# Save clean file
df_clean.to_excel("FAIR_instructions_with_parsed_columns.xlsx", index=False)

         1-Suggested file formats for each data type  \
0  {'Demographic and Clinical Data': 'CSV (Comma-...   
1  {'Genomic raw data': 'FASTQ (raw reads), BAM/C...   
2  For single-cell matrices (ATAC, RNA, DNAm): us...   
3  {'MRI images': 'NIfTI (.nii or .nii.gz) is rec...   

  2-Tools for converting data to the suggested file formats  \
0  {'Demographic and Clinical Data': 'Microsoft E...          
1  {'Genomic data': 'bcl2fastq (Illumina), samtoo...          
2  Seurat (R), Scanpy (Python), and Bioconductor ...          
3  {'MRI images': 'dcm2niix is a widely used tool...          

  3-Suggested standard structure to organize the dataset  
0  BIDS (Brain Imaging Data Structure) is the rec...      
1  Follow the NIMH Data Archive (NDA) data struct...      
2  Recommended approach: Use the Human Cell Atlas...      
3  The Brain Imaging Data Structure (BIDS) is the...      


In [9]:
df_clean

,link,id,fundingSource,institute,isHumanStudy,consentDescription,element_1A,prompt,gpt_answer,1-Suggested file formats for each data type,2-Tools for converting data to the suggested file formats,3-Suggested standard structure to organize the dataset,4-Tools for organizing the dataset in the suggested structure,5-Metadata and information that should be provided with the data,6-Suggested standard structure and file format for metadata,7-Tools for preparing metadata in the suggested structure and file format,8-Suggested de-identification approach,9-Tools to implement the suggested de-identification approach,10-Suggested repositories for sharing the dataset,11-Suggested license for sharing the dataset
0,https://grants.nih.gov/sites/default/files/flm...,1,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,"Demographic, clinical, and MRI, 1 H fMRS and f...",You are a FAIR data expert helping researchers...,"{\n ""1-Suggested file formats for each data t...",{'Demographic and Clinical Data': 'CSV (Comma-...,{'Demographic and Clinical Data': 'Microsoft E...,BIDS (Brain Imaging Data Structure) is the rec...,"BIDS Starter Kit, HeuDiConv, bidskit, and the ...",Comprehensive metadata should include: (1) Dat...,BIDS-compliant JSON files for imaging metadata...,"BIDS Validator (web or command-line), BIDS Sta...",De-identification is required for all human su...,pydeface or mri_deface for defacing MRI images...,The NIMH Data Archive (NDA) is the required re...,Data shared via NDA is subject to NDA's Data U...
1,https://grants.nih.gov/sites/default/files/flm...,2,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,Our genomic study will be registered with dbGa...,You are a FAIR data expert helping researchers...,"{\n ""1-Suggested file formats for each data t...","{'Genomic raw data': 'FASTQ (raw reads), BAM/C...","{'Genomic data': 'bcl2fastq (Illumina), samtoo...",Follow the NIMH Data Archive (NDA) data struct...,"NDA Data Submission Tool (NDA Upload Tool), ND...",Comprehensive metadata including study descrip...,NDA requires metadata in their standardized da...,"NDA Data Dictionary Tool, NDA web-based metada...",De-identification is required for all human su...,NDA GUID Tool for generating subject identifie...,Raw and processed genomic data: dbGaP (require...,Data should be shared under controlled access ...
2,https://grants.nih.gov/sites/default/files/flm...,3,NIH,National Institute of Mental Health (NIMH),no,no,"As detailed in the Research Strategy Section, ...",You are a FAIR data expert helping researchers...,"{\n ""1-Suggested file formats for each data t...","For single-cell matrices (ATAC, RNA, DNAm): us...","Seurat (R), Scanpy (Python), and Bioconductor ...",Recommended approach: Use the Human Cell Atlas...,The HCA Data Coordination Platform (DCP) tools...,Provide a comprehensive README file describing...,Recommended approach: Use the Minimum Informat...,"HCA metadata submission tools, DataCite Metada...",De-identification is not required for mouse da...,"For mouse data, no de-identification tools are...","For mouse single-cell omics data, deposit in t...",Recommended approach: Use the Creative Commons...
3,https://grants.nih.gov/sites/default/files/flm...,4,NIH,National Institute of Mental Health (NIMH),yes,All research participants will be consented fo...,The data to be shared will include MRI images ...,You are a FAIR data expert helping researchers...,"{\n ""1-Suggested file formats for each data t...",{'MRI images': 'NIfTI (.nii or .nii.gz) is rec...,{'MRI images': 'dcm2niix is a widely used tool...,The Brain Imaging Data Structure (BIDS) is the...,"BIDS Starter Kit (templates and guides), BIDSc...",Provide a comprehensive README file describing...,"Use the BIDS metadata structure, which include...","BIDS Starter Kit (metadata templates), BIDS Va...",De-identification is required for all human su...,"For MRI: pydeface, mri_deface

In [10]:
print(len(df_clean.columns))
print(df_clean.columns.tolist())

20
['link', 'id', 'fundingSource', 'institute', 'isHumanStudy', 'consentDescription', 'element_1A', 'prompt', 'gpt_answer', '1-Suggested file formats for each data type', '2-Tools for converting data to the suggested file formats', '3-Suggested standard structure to organize the dataset', '4-Tools for organizing the dataset in the suggested structure', '5-Metadata and information that should be provided with the data', '6-Suggested standard structure and file format for metadata', '7-Tools for preparing metadata in the suggested structure and file format', '8-Suggested de-identification approach', '9-Tools to implement the suggested de-identification approach', '10-Suggested repositories for sharing the dataset', '11-Suggested license for sharing the dataset']
